# 02 — Add Match Metadata and Chronological Order
Load match metadata, merge it into extracted shot events, derive clean chronological fields,
and save chronologically ordered datasets for later feature engineering.

In [13]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

## Step 0: Path validation

In [14]:
DATA_DIR  = Path("../data")
MATCH_DIR = DATA_DIR / "wyscout-data" / "matches"

assert (DATA_DIR / "wyscout_shots.parquet").exists(),       "Run 01_extract_shots first"
assert (DATA_DIR / "wyscout_train_shots.parquet").exists(), "Run 01_extract_shots first"
assert (DATA_DIR / "wyscout_test_shots.parquet").exists(),  "Run 01_extract_shots first"
assert MATCH_DIR.exists(),                                   f"Missing: {MATCH_DIR}"
assert (MATCH_DIR / "matches_England.json").exists(),       "England match file not found"
print("Paths OK")

Paths OK


## Step 1: Load shot tables from notebook 01

In [ ]:
shots_orig  = pd.read_parquet(DATA_DIR / "wyscout_shots.parquet")
train_orig  = pd.read_parquet(DATA_DIR / "wyscout_train_shots.parquet")
test_orig   = pd.read_parquet(DATA_DIR / "wyscout_test_shots.parquet")

assert len(shots_orig) > 0
assert len(shots_orig) == len(train_orig) + len(test_orig), "Combined ≠ train + test"

EXPECTED_SHOT_COLS = {
    "league", "matchId", "playerId", "teamId",
    "eventSec", "matchPeriod", "eventName", "subEventName",
    "x", "y", "is_goal", "is_penalty", "is_direct_free_kick",
}
assert EXPECTED_SHOT_COLS.issubset(shots_orig.columns), \
    f"Missing columns: {EXPECTED_SHOT_COLS - set(shots_orig.columns)}"

# Normalise flag dtypes immediately after load
for df in [shots_orig, train_orig, test_orig]:
    df["is_penalty"]          = df["is_penalty"].astype("int8")
    df["is_direct_free_kick"] = df["is_direct_free_kick"].astype("int8")

print(f"Combined: {len(shots_orig):,}  |  Train: {len(train_orig):,}  |  Test: {len(test_orig):,}")

## Step 2: Helper functions

In [16]:
# matchPeriod → integer sort key
PERIOD_ORDER = {"1H": 1, "2H": 2, "E1": 3, "E2": 4, "P": 5}

def period_sort_key(period_series: pd.Series) -> pd.Series:
    return period_series.map(PERIOD_ORDER)


# Continuous match-time ordering key (not true ball-in-play time — resets at period boundaries).
# Vectorised: map each period to its offset, then add eventSec.
PERIOD_OFFSETS = {"1H": 0, "2H": 45*60, "E1": 90*60, "E2": 105*60, "P": 120*60}

def compute_match_seconds(df: pd.DataFrame) -> pd.Series:
    """Return seconds_from_match_start for every row (vectorised)."""
    offsets = df["matchPeriod"].map(PERIOD_OFFSETS)
    return offsets + df["eventSec"]


def build_match_df(raw_matches: list, league: str) -> pd.DataFrame:
    rows = []
    for m in raw_matches:
        rows.append({
            "league":               league,
            "matchId":              m["wyId"],
            "match_date_utc_raw":   m.get("dateutc"),
            "match_date_local_raw": m.get("date"),
            "gameweek":             m.get("gameweek"),
            "roundId":              m.get("roundId"),
            "seasonId":             m.get("seasonId"),
            "competitionId":        m.get("competitionId"),
            "match_label":          m.get("label"),
            "match_status":         m.get("status"),
            "match_duration":       m.get("duration"),
            "winner":               m.get("winner"),
        })
    return pd.DataFrame(rows)

## Step 3: Load and flatten match metadata per league

In [17]:
leagues = ["England", "France", "Germany", "Italy", "Spain"]
match_dfs = {}

for league in leagues:
    with open(MATCH_DIR / f"matches_{league}.json", encoding="utf-8") as f:
        raw = json.load(f)
    match_dfs[league] = build_match_df(raw, league)
    print(f"{league}: {len(match_dfs[league]):,} matches")

matches_df = pd.concat(match_dfs.values(), ignore_index=True)

# dateutc strings are naive (no tz marker) but represent UTC by Wyscout convention.
# utc=True tells pandas to attach UTC after parsing rather than requiring it in the string.
matches_df["match_datetime_utc"] = pd.to_datetime(
    matches_df["match_date_utc_raw"], utc=True, errors="coerce"
)
matches_df["match_date"] = matches_df["match_datetime_utc"].dt.date

# Sanity checks
assert matches_df["matchId"].notna().all(), "Null matchId in matches"
assert matches_df["match_datetime_utc"].notna().all(), "Null datetime in matches"
assert not matches_df[["league", "matchId"]].duplicated().any(), "Duplicate (league, matchId) in matches"

print(f"\nTotal matches: {len(matches_df):,}")

England: 380 matches
France: 380 matches
Germany: 306 matches
Italy: 380 matches
Spain: 380 matches

Total matches: 1,826


## Step 4: Merge match metadata into shots

In [18]:
shots = shots_orig.merge(
    matches_df.drop(columns=["match_date_utc_raw"]),
    on=["league", "matchId"],
    how="left",
    validate="many_to_one",
)

assert len(shots) == len(shots_orig), "Row count changed after merge"
assert shots["match_datetime_utc"].notna().all(), "Unmatched shots (null match_datetime_utc)"
# Hard requirement: gameweek must be present on every shot — needed for rolling form features downstream.
assert shots["gameweek"].notna().all(), "Null gameweek after merge — check match files"

print(f"Merged shots: {len(shots):,}")
shots[["league", "matchId", "match_datetime_utc", "gameweek", "match_label"]].head()

Merged shots: 40,461


,league,matchId,match_datetime_utc,gameweek,match_label
0,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"
1,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"
2,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"
3,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"
4,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"


## Step 5: Build chronological ordering fields

> `seconds_from_match_start` is a sort key only — it is NOT true ball-in-play time.  
> Wyscout's `eventSec` resets at the start of each period.

In [19]:
shots["period_sort_key"] = period_sort_key(shots["matchPeriod"])
assert shots["period_sort_key"].notna().all(), (
    f"Unexpected matchPeriod values: {shots[shots['period_sort_key'].isna()]['matchPeriod'].unique()}"
)

shots["seconds_from_match_start"] = compute_match_seconds(shots)

shots = shots.sort_values(
    ["match_datetime_utc", "matchId", "period_sort_key", "eventSec", "playerId", "teamId"],
    ignore_index=True,
)

shots["shot_sequence_in_match"]      = shots.groupby(["league", "matchId"]).cumcount() + 1
shots["shot_sequence_team_in_match"] = shots.groupby(["league", "matchId", "teamId"]).cumcount() + 1

# Rebuild train/test from sorted combined df to guarantee consistency
train_shots = shots[shots["league"] != "England"].copy()
test_shots  = shots[shots["league"] == "England"].copy()

assert set(train_shots["league"].unique()) == {"France","Germany","Italy","Spain"}, "Wrong leagues in train"
assert set(test_shots["league"].unique())  == {"England"}, "Wrong leagues in test"

print(f"Train: {len(train_shots):,}  |  Test: {len(test_shots):,}")

Train: 32,010  |  Test: 8,451


## Step 6: Sanity checks

In [ ]:
ID_COLS = ["league", "matchId", "playerId", "teamId", "eventSec", "matchPeriod"]
assert not shots[ID_COLS].duplicated().any(), "Duplicate shot identities after merge"

for col in ["match_datetime_utc", "period_sort_key", "seconds_from_match_start"]:
    assert shots[col].notna().all(), f"Null values in {col}"

assert shots["seconds_from_match_start"].ge(0).all(), "Negative seconds_from_match_start"
assert shots["period_sort_key"].isin([1,2,3,4,5]).all(), "Unexpected period_sort_key values"

# Attempt-type flag checks
assert shots["is_penalty"].isin([0, 1]).all(), "is_penalty not binary"
assert shots["is_direct_free_kick"].isin([0, 1]).all(), "is_direct_free_kick not binary"
assert not ((shots["is_penalty"] == 1) & (shots["is_direct_free_kick"] == 1)).any(), \
    "A row has both is_penalty and is_direct_free_kick = 1"

# Event/subevent pair consistency
assert (shots.loc[shots["is_penalty"] == 1, "eventName"] == "Free Kick").all(), \
    "Penalty row has unexpected eventName"
assert (shots.loc[shots["is_penalty"] == 1, "subEventName"] == "Penalty").all(), \
    "Penalty row has unexpected subEventName"
assert (shots.loc[shots["is_direct_free_kick"] == 1, "eventName"] == "Free Kick").all(), \
    "Direct FK shot row has unexpected eventName"
assert (shots.loc[shots["is_direct_free_kick"] == 1, "subEventName"] == "Free kick shot").all(), \
    "Direct FK shot row has unexpected subEventName"
assert (shots.loc[
    (shots["is_penalty"] == 0) & (shots["is_direct_free_kick"] == 0), "eventName"
] == "Shot").all(), "Standard Shot/Shot rows have unexpected eventName"

print("All merge-quality checks passed")
print(shots[["match_datetime_utc","gameweek","period_sort_key","seconds_from_match_start"]].describe())

In [21]:
assert shots["match_datetime_utc"].is_monotonic_increasing, "Shots not sorted by match_datetime_utc"

# Within each match, period_sort_key must be non-decreasing
within_match_period_ok = (
    shots.groupby(["league","matchId"])["period_sort_key"]
         .apply(lambda s: s.diff().fillna(0).ge(0).all())
         .all()
)
assert within_match_period_ok, "period_sort_key not non-decreasing within a match"

# shot_sequence_in_match must run 1..n with no gaps and no duplicates
seq = shots.groupby(["league","matchId"])["shot_sequence_in_match"].agg(["min","max","count"])
assert (seq["min"] == 1).all(), "shot_sequence_in_match does not start at 1 for every match"
assert (seq["max"] == seq["count"]).all(), "Gaps or duplicates in shot_sequence_in_match"

print(f"Earliest match: {shots['match_datetime_utc'].min()}")
print(f"Latest match:   {shots['match_datetime_utc'].max()}")
print(f"Matches in shots: {shots[['league','matchId']].drop_duplicates().shape[0]:,}")

Earliest match: 2017-08-04 18:45:00+00:00
Latest match:   2018-05-20 18:45:00+00:00
Matches in shots: 1,826


In [22]:
summary = shots.groupby("league").agg(
    n_matches=("matchId", "nunique"),
    n_shots=("matchId", "count"),
    min_date=("match_date", "min"),
    max_date=("match_date", "max"),
    min_gameweek=("gameweek", "min"),
    max_gameweek=("gameweek", "max"),
)
display(summary)

,n_matches,n_shots,min_date,max_date,min_gameweek,max_gameweek
league,,,,,,
England,380,8451,2017-08-11,2018-05-13,1,38
France,380,8327,2017-08-04,2018-05-19,1,38
Germany,306,6898,2017-08-18,2018-05-12,1,34
Italy,380,8806,2017-08-19,2018-05-20,1,38
Spain,380,7979,2017-08-18,2018-05-20,1,38


## Step 7: Save outputs

In [ ]:
SAVE_COLS = [
    # Shot identity / event fields
    "league", "matchId", "playerId", "teamId",
    "eventSec", "matchPeriod", "eventName", "subEventName",
    "x", "y", "is_goal", "is_penalty", "is_direct_free_kick",
    # Match metadata (scalar only)
    "match_datetime_utc", "match_date", "match_date_local_raw",
    "gameweek", "roundId", "seasonId", "competitionId",
    "match_label", "match_status", "match_duration", "winner",
    # Chronological ordering fields
    "period_sort_key", "seconds_from_match_start",
    "shot_sequence_in_match", "shot_sequence_team_in_match",
]

shots[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_shots_with_matches.parquet", index=False)
train_shots[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_train_shots_with_matches.parquet", index=False)
test_shots[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_test_shots_with_matches.parquet", index=False)
shots[SAVE_COLS].head(100).to_csv(DATA_DIR / "wyscout_shots_with_matches_sample.csv", index=False)

print(f"Saved {len(shots):,} combined  → wyscout_shots_with_matches.parquet")
print(f"Saved {len(train_shots):,} train    → wyscout_train_shots_with_matches.parquet")
print(f"Saved {len(test_shots):,} test     → wyscout_test_shots_with_matches.parquet")
print("Saved 100-row sample        → wyscout_shots_with_matches_sample.csv")

In [ ]:
check = pd.read_parquet(DATA_DIR / "wyscout_shots_with_matches.parquet")
assert check.shape == shots[SAVE_COLS].shape, "Shape mismatch on reload"
assert set(SAVE_COLS) == set(check.columns), "Column mismatch on reload"
assert pd.api.types.is_datetime64_any_dtype(check["match_datetime_utc"]), \
    "match_datetime_utc not datetime on reload"
assert check["is_penalty"].isin([0, 1]).all(), "is_penalty not binary on reload"
assert check["is_direct_free_kick"].isin([0, 1]).all(), "is_direct_free_kick not binary on reload"
print(check.shape)
check.head()